# NAIP Inference — Walmart Parking Lot Occupancy

Runs the fine-tuned ResNet34 U-Net segmentation model on all images in `Data/naip_images`.

**Output** → `Data/naip_occupancy_results.csv` with columns:

| Column | Description |
|---|---|
| `store_id` | Integer extracted from the filename prefix (e.g. `34` from `34_2019-07-01.png`) |
| `date` | ISO date string extracted from the filename (e.g. `2019-07-01`) |
| `year` | Integer year derived from `date` |
| `occupancy` | Fraction of detected parking pixels that are occupied (0–1) |

**Analysis** (Section 3):
- Average occupancy per year
- Number of images per year

## 1 · Setup

In [ ]:
import re
from pathlib import Path

import albumentations as A
import numpy as np
import pandas as pd
import torch
from albumentations.pytorch import ToTensorV2
from PIL import Image
import segmentation_models_pytorch as smp

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_ROOT      = Path("../Data")
NAIP_DIR       = DATA_ROOT / "naip_images"
CHECKPOINT_DIR = Path("../checkpoints")
OUTPUT_CSV     = DATA_ROOT / "naip_occupancy_results.csv"

FT_CKPT   = CHECKPOINT_DIR / "resnet34_unet_walmart_ft.pth"
BASE_CKPT = CHECKPOINT_DIR / "resnet34_unet_best.pth"
CKPT_PATH = FT_CKPT if FT_CKPT.exists() else BASE_CKPT

# ── Model config (must match training) ────────────────────────────────────
IN_CHANNELS = 3
NUM_CLASSES = 3
IMG_SIZE    = 256

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# ── Device ─────────────────────────────────────────────────────────────────
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

ckpt_label = "fine-tuned" if CKPT_PATH == FT_CKPT else "base PKLot (fine-tuned not found)"
print(f"Device     : {DEVICE}")
print(f"Checkpoint : {CKPT_PATH}  ({ckpt_label})")
print(f"Images dir : {NAIP_DIR.resolve()}")

In [ ]:
model = smp.Unet(
    encoder_name    = "resnet34",
    encoder_weights = None,
    in_channels     = IN_CHANNELS,
    classes         = NUM_CLASSES,
    activation      = None,
)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print(f"Model loaded from {CKPT_PATH.name}, eval mode on {DEVICE}")

## 2 · Discover & Parse Images

Filename format: `{store_id}_{YYYY-MM-DD}.png`  
- `store_id` = integer before the first underscore  
- `date`     = ISO date string after the first underscore

In [ ]:
all_paths = sorted(NAIP_DIR.glob("*.png"))
print(f"Found {len(all_paths)} PNG files in {NAIP_DIR}\n")

records = []
skipped = []
for p in all_paths:
    m = re.match(r"^(\d+)_(\d{4}-\d{2}-\d{2})", p.stem)
    if m:
        records.append({"path": p, "store_id": int(m.group(1)), "date": m.group(2)})
    else:
        skipped.append(p.name)

print(f"Parsed successfully : {len(records)}")
if skipped:
    print(f"Skipped (no match)  : {skipped}")

# Preview first few
for r in records[:5]:
    print(f"  store={r['store_id']:>5}  date={r['date']}  file={r['path'].name}")

## 2 · Run Inference

For each image:
1. Resize to 256×256, normalise with ImageNet stats
2. Run the model (argmax over class logits)
3. Compute **occupancy** = `occupied_pixels / (empty_pixels + occupied_pixels)`

Class indices: `0` = background, `1` = space-empty, `2` = space-occupied.

In [ ]:
preprocess = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

def run_inference(path: Path) -> float:
    """Return occupancy rate (0–1) for a single image."""
    img     = np.array(Image.open(path).convert("RGB"))
    resized = np.array(Image.fromarray(img).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
    tensor  = preprocess(image=resized)["image"].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(tensor).argmax(dim=1).squeeze(0).cpu().numpy()  # (256, 256)

    empty    = int((pred == 1).sum())
    occupied = int((pred == 2).sum())
    parking  = empty + occupied
    return occupied / parking if parking > 0 else 0.0


rows = []
n    = len(records)
print(f"Running inference on {n} images …")

for i, rec in enumerate(records, 1):
    occ = run_inference(rec["path"])
    rows.append({"store_id": rec["store_id"], "date": rec["date"], "occupancy": occ})
    if i % 50 == 0 or i == n:
        print(f"  [{i:>3}/{n}] last: store={rec['store_id']}  {rec['date']}  occ={occ:.3f}")

print("\nDone.")

In [ ]:
df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df = df.sort_values(["store_id", "date"]).reset_index(drop=True)

# Save
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df)} rows → {OUTPUT_CSV.resolve()}\n")
df.head(10)

## 3 · Analysis

### 3a · Average Occupancy per Year

Mean occupancy rate across all stores and images grouped by calendar year.

In [ ]:
import matplotlib.pyplot as plt

avg_occ = (
    df.groupby("year")["occupancy"]
    .agg(mean_occupancy="mean", std_occupancy="std", n_images="count")
    .reset_index()
)
avg_occ["mean_pct"] = avg_occ["mean_occupancy"] * 100
avg_occ["std_pct"]  = avg_occ["std_occupancy"]  * 100

print("Average occupancy per year\n")
print(f"{'Year':>6}  {'Mean occ':>10}  {'Std':>8}  {'N images':>9}")
print("─" * 40)
for _, row in avg_occ.iterrows():
    print(f"{int(row.year):>6}  {row.mean_pct:>9.1f}%  {row.std_pct:>7.1f}%  {int(row.n_images):>9}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(avg_occ["year"], avg_occ["mean_pct"],
       yerr=avg_occ["std_pct"], capsize=4,
       color="steelblue", edgecolor="white", alpha=0.85)
ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("Mean occupancy (%)", fontsize=11)
ax.set_title("Average Parking Occupancy per Year — All NAIP Stores", fontsize=13)
ax.set_xticks(avg_occ["year"])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.grid(axis="y", linestyle="--", alpha=0.4)
for x, y in zip(avg_occ["year"], avg_occ["mean_pct"]):
    ax.text(x, y + avg_occ.loc[avg_occ.year == x, "std_pct"].values[0] + 0.5,
            f"{y:.1f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

### 3b · Number of Images per Year

In [ ]:
img_count = df.groupby("year").size().reset_index(name="n_images")

print("Number of images per year\n")
print(f"{'Year':>6}  {'N images':>9}")
print("─" * 20)
for _, row in img_count.iterrows():
    print(f"{int(row.year):>6}  {int(row.n_images):>9}")
print(f"{'Total':>6}  {img_count.n_images.sum():>9}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(img_count["year"], img_count["n_images"],
       color="darkorange", edgecolor="white", alpha=0.85)
ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("Number of images", fontsize=11)
ax.set_title("Number of NAIP Images per Year", fontsize=13)
ax.set_xticks(img_count["year"])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{int(y)}"))
ax.grid(axis="y", linestyle="--", alpha=0.4)
for x, y in zip(img_count["year"], img_count["n_images"]):
    ax.text(x, y + 0.3, str(int(y)), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()